# DEH 30-Day PySpark Challenge
## Day 25 — Practice Set 4: String & Date Functions

**Instructions:**
1. `File → Save a copy in Drive` first
2. SQL then PySpark for each problem
3. Reference solutions follow each problem

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder.appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id", StringType(), True), StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True), StructField("order_date", DateType(), True),
    StructField("quantity", IntegerType(), True), StructField("unit_price", DoubleType(), True),
    StructField("discount_pct", IntegerType(), True), StructField("status", StringType(), True),
    StructField("payment_method", StringType(), True), StructField("region", StringType(), True)
])
customers_schema = StructType([
    StructField("customer_id", StringType(), True), StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True), StructField("email", StringType(), True),
    StructField("city", StringType(), True), StructField("state", StringType(), True),
    StructField("country", StringType(), True), StructField("signup_date", DateType(), True),
    StructField("segment", StringType(), True)
])

orders_df    = spark.read.option("header","true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
customers_df = spark.read.option("header","true").schema(customers_schema).csv(f"s3a://{BUCKET}/data/customers.csv")
orders_df.createOrReplaceTempView("orders")
customers_df.createOrReplaceTempView("customers")
print(f"Orders: {orders_df.count()} | Customers: {customers_df.count()}")

---
## Problem 1 (Easy) — Standardize customer names

Create `full_name` combining first and last name in Title Case, trimmed, single space.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 1

In [ ]:
spark.sql("""
    SELECT customer_id,
           INITCAP(TRIM(CONCAT_WS(' ', first_name, last_name))) AS full_name
    FROM customers
""").show(5)

In [ ]:
customers_df.select(
    "customer_id",
    F.initcap(F.trim(F.concat_ws(" ", F.col("first_name"), F.col("last_name")))).alias("full_name")
).show(5)

---
## Problem 2 (Easy) — Extract email domain, flag gmail

Extract domain from email. is_gmail = true if domain contains 'gmail'.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 2

In [ ]:
spark.sql("""
    SELECT customer_id, email,
           SUBSTRING_INDEX(email, '@', -1) AS domain,
           SUBSTRING_INDEX(email, '@', -1) LIKE '%gmail%' AS is_gmail
    FROM customers
""").show(5, truncate=False)

In [ ]:
customers_df.select(
    "customer_id", "email",
    F.split(F.col("email"), "@")[1].alias("domain"),
    F.split(F.col("email"), "@")[1].contains("gmail").alias("is_gmail")
).show(5, truncate=False)

---
## Problem 3 (Medium) — Orders placed on weekends

Find orders on Saturday or Sunday. Return order_id, order_date, day_name. Count them.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 3

Remember: `dayofweek()` returns 1=Sunday, 7=Saturday.

In [ ]:
spark.sql("""
    SELECT order_id, order_date, DATE_FORMAT(order_date, 'EEEE') AS day_name
    FROM orders
    WHERE DAYOFWEEK(order_date) IN (1, 7)
""").show()

spark.sql("""
    SELECT COUNT(*) AS weekend_orders FROM orders WHERE DAYOFWEEK(order_date) IN (1, 7)
""").show()

In [ ]:
weekend_orders = orders_df.withColumn("day_name", F.date_format("order_date", "EEEE")) \
    .filter(F.dayofweek("order_date").isin(1, 7)) \
    .select("order_id", "order_date", "day_name")

weekend_orders.show()
print(f"Weekend orders: {weekend_orders.count()}")

---
## Problem 4 (Medium) — Early Adopter Purchase flag

Orders within 90 days of signup_date = 'Early Adopter Purchase', else 'Regular Purchase'.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 4

In [ ]:
spark.sql("""
    SELECT o.order_id, c.customer_id, o.order_date, c.signup_date,
           DATEDIFF(o.order_date, c.signup_date) AS days_since_signup,
           CASE WHEN DATEDIFF(o.order_date, c.signup_date) <= 90
                THEN 'Early Adopter Purchase' ELSE 'Regular Purchase' END AS purchase_type
    FROM orders o
    INNER JOIN customers c ON o.customer_id = c.customer_id
    ORDER BY days_since_signup
""").show(10)

In [ ]:
orders_df.join(customers_df, on="customer_id", how="inner") \
    .withColumn("days_since_signup", F.datediff(F.col("order_date"), F.col("signup_date"))) \
    .withColumn(
        "purchase_type",
        F.when(F.col("days_since_signup") <= 90, "Early Adopter Purchase").otherwise("Regular Purchase")
    ).select("order_id", "customer_id", "order_date", "signup_date", "days_since_signup", "purchase_type") \
    .orderBy("days_since_signup") \
    .show(10)

---
## Problem 5 (Hard) — Mask email addresses

Show first 2 chars of username + asterisks + full domain. No UDF.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 5

In [ ]:
spark.sql("""
    SELECT email,
           CONCAT(
               SUBSTRING(SUBSTRING_INDEX(email, '@', 1), 1, 2),
               '****@',
               SUBSTRING_INDEX(email, '@', -1)
           ) AS masked_email
    FROM customers
""").show(5, truncate=False)

In [ ]:
customers_df.withColumn("username", F.split(F.col("email"), "@")[0]) \
    .withColumn("domain", F.split(F.col("email"), "@")[1]) \
    .withColumn(
        "masked_email",
        F.concat(F.substring(F.col("username"), 1, 2), F.lit("****@"), F.col("domain"))
    ).select("email", "masked_email") \
    .show(5, truncate=False)

---
## Problem 6 (Hard) — Fiscal quarter label

Fiscal year starts April. Generate labels like 'FY2023-Q3' from order_date.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 6

Apr-Jun=Q1, Jul-Sep=Q2, Oct-Dec=Q3, Jan-Mar=Q4 of the PREVIOUS fiscal year (since fiscal year started the prior April).

In [ ]:
spark.sql("""
    SELECT order_id, order_date,
           CASE
               WHEN MONTH(order_date) BETWEEN 4 AND 6  THEN CONCAT('FY', YEAR(order_date), '-Q1')
               WHEN MONTH(order_date) BETWEEN 7 AND 9  THEN CONCAT('FY', YEAR(order_date), '-Q2')
               WHEN MONTH(order_date) BETWEEN 10 AND 12 THEN CONCAT('FY', YEAR(order_date), '-Q3')
               ELSE CONCAT('FY', YEAR(order_date) - 1, '-Q4')
           END AS fiscal_quarter
    FROM orders
    ORDER BY order_date
""").show(15)

In [ ]:
orders_df.withColumn(
    "fiscal_quarter",
    F.when((F.month("order_date") >= 4) & (F.month("order_date") <= 6),
           F.concat(F.lit("FY"), F.year("order_date"), F.lit("-Q1")))
     .when((F.month("order_date") >= 7) & (F.month("order_date") <= 9),
           F.concat(F.lit("FY"), F.year("order_date"), F.lit("-Q2")))
     .when((F.month("order_date") >= 10) & (F.month("order_date") <= 12),
           F.concat(F.lit("FY"), F.year("order_date"), F.lit("-Q3")))
     .otherwise(F.concat(F.lit("FY"), F.year("order_date") - 1, F.lit("-Q4")))
).select("order_id", "order_date", "fiscal_quarter").orderBy("order_date").show(15)

---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*